In [1]:
import torch
from torch import nn, optim
import torch.nn.functional as F
import torch.utils.data as data
import joblib
import numpy as np
import os
import glob
import random
import gc
from tqdm import tqdm
from python_core.general import info, warning, error, debug

from models import Model

device = "cuda"
seed = 1
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

hours = 24 * 7
max_len = 100

In [2]:
global_info = joblib.load("Gowalla_thinned/global_info.pkl")
mat2s = torch.FloatTensor(global_info["mat2s"])
u_max = global_info["u_max"]
l_max = global_info["l_max"]

info(f"u_max: {u_max}, l_max: {l_max}")
info(f"mat2s shape: {mat2s.shape}")

train_chunk_files = sorted(glob.glob("Gowalla_thinned/chunks/train_stan_chunk_*.pkl"))

# 讀取第一個分塊來估算特徵極值 (ex)
first_chunk = joblib.load(train_chunk_files[0])
mat1_sample = first_chunk[1]  # mat1 (N, M, M, 2)
ex = (
    mat1_sample[:, :, :, 0].max(),
    mat1_sample[:, :, :, 0].min(),
    mat1_sample[:, :, :, 1].max(),
    mat1_sample[:, :, :, 1].min(),
)

info(f"ex: {ex}")
del first_chunk, mat1_sample
gc.collect()

INFO   u_max: 45199, l_max: 29511
INFO   mat2s shape: torch.Size([29511, 29511])
INFO   ex: (np.float32(12063.293), np.float32(0.0), np.float32(204.0), np.float32(0.0))


70

In [3]:
def sampling_prob(prob, label, num_neg, l_m):
    # prob: (N, L), L = l_m
    num_label = prob.shape[0]
    label = label.view(-1)
    init_label = np.linspace(0, num_label-1, num_label)
    init_prob = torch.zeros(size=(num_label, num_neg+len(label)))

    random_ig = random.sample(range(0, l_m), num_neg)
    
    # 確保負樣本中沒有抽到正確答案
    label_set = set(label.cpu().tolist())
    while len(label_set.intersection(random_ig)) != 0:
        random_ig = random.sample(range(0, l_m), num_neg)

    # 組合機率分佈：前半為正樣本，後半為負樣本
    for k in range(num_label):
        for i in range(num_neg + len(label)):
            if i < len(label):
                init_prob[k, i] = prob[k, label[i]]
            else:
                init_prob[k, i] = prob[k, random_ig[i-len(label)]]

    return torch.FloatTensor(init_prob), torch.LongTensor(init_label)

class ChunkDataset(data.Dataset):
    def __init__(self, chunk_data):
        self.traj, self.mat1, self.vec, self.label, self.length = chunk_data

    def __getitem__(self, index):
        traj = self.traj[index].to(device)
        mats1 = torch.FloatTensor(self.mat1[index]).to(device)
        vector = torch.FloatTensor(self.vec[index]).to(device)
        label = self.label[index].to(device) - 1 # cross entropy 由 0 開始算
        length = self.length[index].to(device)
        return traj, mats1, vector, label, length

    def __len__(self):
        return len(self.traj)

In [4]:
embed_dim = 50
learning_rate = 3e-3
num_neg = 10

model = Model(t_dim=hours+1, l_dim=l_max+1, u_dim=u_max+1, embed_dim=embed_dim, ex=ex, dropout=0)
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=0)

In [5]:
max_len = 100
num_epochs = 1  
batch_size = 1  

model.train()

for epoch in range(num_epochs):
    print(f"\n--- 訓練 Epoch {epoch+1}/{num_epochs} ---")
    total_loss = 0.0
    step_count = 0
    
    for chunk_idx, chunk_file in enumerate(train_chunk_files):
        chunk_data = joblib.load(chunk_file)
        dataset = ChunkDataset(chunk_data)
        loader = data.DataLoader(dataset=dataset, batch_size=batch_size, shuffle=True)
        
        chunk_loss = 0.0
        
        for item in tqdm(loader, desc=f"Chunk {chunk_idx+1}/{len(train_chunk_files)}", leave=False):
            person_input, person_m1, person_m2t, person_label, person_traj_len = item
            traj_len_val = person_traj_len[0].item()
            
            # 【修復核心 1】：強制將動態長度的軌跡補零至 max_len (100)
            current_m = person_input.shape[1]
            if current_m < max_len:
                person_input = F.pad(person_input, (0, 0, 0, max_len - current_m), mode='constant', value=0)
            
            for mask_len in range(1, traj_len_val + 1):
                
                if mask_len > traj_len_val - 2: 
                    break

                # 【修復核心 2】：恢復原本的寫死維度遮罩 (維持 100 長度)
                input_mask = torch.zeros((batch_size, max_len, 3), dtype=torch.long).to(device)
                m1_mask = torch.zeros((batch_size, max_len, max_len, 2), dtype=torch.float32).to(device)
                
                input_mask[:, :mask_len] = 1
                m1_mask[:, :mask_len, :mask_len] = 1.0

                train_input = person_input * input_mask
                train_m1 = person_m1 * m1_mask
                train_len = torch.zeros(size=(batch_size,), dtype=torch.long).to(device) + mask_len

                # 【修復核心 3】：確保目標時間差向量 train_m2t 也是 100 的長度
                if mask_len < traj_len_val:
                    train_label = person_input[:, mask_len, 1] - 1
                    # 計算當前預測點與歷史 100 個位置 (超出 mask_len 的部分為 0) 的絕對時間差
                    train_m2t = torch.abs(person_input[:, mask_len, 2:3] - person_input[:, :, 2]).float()
                else:
                    train_label = person_label
                    # 前處理產生的 person_m2t 已經是 100 的長度
                    train_m2t = person_m2t 

                # 前向傳播
                prob = model(train_input, train_m1, mat2s, train_m2t, train_len)

                prob_sample, label_sample = sampling_prob(prob, train_label, num_neg, l_max)
                loss = F.cross_entropy(prob_sample.to(device), label_sample.to(device))

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                chunk_loss += loss.item()
                total_loss += loss.item()
                step_count += 1
                
        # 記憶體清理
        del chunk_data, dataset, loader
        torch.cuda.empty_cache()
        gc.collect()
        
        print(f"Chunk {chunk_idx+1} 完成，當前平均 Loss: {chunk_loss/(step_count+1e-9):.4f}")

    print(f"Epoch {epoch+1} 總結平均 Loss: {total_loss / step_count:.4f}")


--- 訓練 Epoch 1/1 ---


Chunk 1 完成，當前平均 Loss: 2.1107


KeyboardInterrupt: 